# DEM (Differential Emission Measure) Analysis Example

Demonstrates how to compute DEM from multi-wavelength AIA observations using the SITES algorithm.

This example shows two approaches:
1. Real AIA data download using sunpy Fido (requires network)
2. Synthetic data for offline testing

Requires: `pip install "egghouse[dem]"`

In [ ]:
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np

# Import DEM module
from egghouse.dem import     get_temperature_response,     get_default_temperatures,     dem_sites,     dem_sites_pixel,     dem_map,     get_emission_measure,     get_mean_temperature,     HAS_AIAPY, 

# Check for sunpy availability
try:
    import astropy.units as u
    from sunpy.map import Map
    from sunpy.net import Fido, attrs as a
    HAS_SUNPY = True
except ImportError:
    HAS_SUNPY = False

# AIA EUV wavelengths for DEM analysis
AIA_WAVELENGTHS = [94, 131, 171, 193, 211, 335]

print(f"aiapy installed: {HAS_AIAPY}")
print(f"sunpy installed: {HAS_SUNPY}")

if not HAS_AIAPY:
    print("\nNote: Using approximate temperature response functions.")
    print("For accurate results, install aiapy: pip install aiapy")

## Helper Functions

In [ ]:
def download_aia_data(
    obs_time: datetime,
    data_dir: str = "./data/aia",
    wavelengths: list = None,
) -> dict:
    """
    Download AIA data from JSOC using sunpy Fido.

    Parameters
    ----------
    obs_time : datetime
        Target observation time.
    data_dir : str
        Directory to save downloaded files.
    wavelengths : list, optional
        List of wavelengths to download. Default: DEM wavelengths.

    Returns
    -------
    dict
        Dictionary with wavelength keys and file paths as values.
    """
    if not HAS_SUNPY:
        raise ImportError("sunpy is required for data download. "
                         "Install with: pip install sunpy")

    if wavelengths is None:
        wavelengths = AIA_WAVELENGTHS

    # Create data directory
    data_path = Path(data_dir)
    data_path.mkdir(parents=True, exist_ok=True)

    # Time range: narrow window around target time
    time_start = obs_time.strftime("%Y-%m-%d %H:%M:%S")
    time_end = (obs_time.replace(second=obs_time.second + 30)).strftime("%Y-%m-%d %H:%M:%S")

    print(f"   Searching for AIA data near {time_start}...")

    files = {}
    for wave in wavelengths:
        print(f"   Downloading {wave} Å...", end=" ", flush=True)

        # Search for data
        result = Fido.search(
            a.Time(time_start, time_end),
            a.Instrument("AIA"),
            a.Wavelength(wave * u.angstrom),
        )

        if len(result) == 0 or len(result[0]) == 0:
            print("not found!")
            continue

        # Fetch first result
        downloaded = Fido.fetch(result[0, 0], path=str(data_path), progress=False)

        if len(downloaded) > 0:
            files[wave] = str(downloaded[0])
            print("OK")
        else:
            print("failed!")

    return files


def load_aia_cube(files: dict, crop_center: tuple = None, crop_size: int = 256) -> dict:
    """
    Load AIA files into image cube for DEM analysis.

    Parameters
    ----------
    files : dict
        Dictionary with wavelength keys and file paths.
    crop_center : tuple, optional
        Center (y, x) for cropping. Default: image center.
    crop_size : int
        Size of cropped region.

    Returns
    -------
    dict
        Dictionary containing:
        - image_cube: (height, width, n_channels)
        - error_cube: estimated errors
        - obs_time: observation time
        - wavelengths: list of wavelengths
    """
    if not HAS_SUNPY:
        raise ImportError("sunpy is required. Install with: pip install sunpy")

    wavelengths = sorted(files.keys())
    maps = []

    for wave in wavelengths:
        m = Map(files[wave])
        maps.append(m)

    # Get image dimensions from first map
    ref_map = maps[0]
    full_height, full_width = ref_map.data.shape

    # Default to center
    if crop_center is None:
        crop_center = (full_height // 2, full_width // 2)

    cy, cx = crop_center
    half = crop_size // 2

    # Crop bounds
    y0 = max(0, cy - half)
    y1 = min(full_height, cy + half)
    x0 = max(0, cx - half)
    x1 = min(full_width, cx + half)

    # Create image cube
    n_channels = len(wavelengths)
    actual_height = y1 - y0
    actual_width = x1 - x0

    image_cube = np.zeros((actual_height, actual_width, n_channels), dtype=np.float32)
    error_cube = np.zeros_like(image_cube)

    for i, (wave, m) in enumerate(zip(wavelengths, maps)):
        # Get data in DN/s
        data = m.data.astype(np.float32)
        exptime = m.exposure_time.to(u.s).value

        # Crop
        cropped = data[y0:y1, x0:x1] / exptime

        # Handle NaN and negative values
        cropped = np.nan_to_num(cropped, nan=0.0, posinf=0.0, neginf=0.0)
        cropped = np.maximum(cropped, 0.0)

        image_cube[:, :, i] = cropped

        # Estimate error (Poisson + readnoise)
        # AIA typical: sqrt(DN) + readnoise (~1.15 DN)
        error_cube[:, :, i] = np.sqrt(np.abs(cropped * exptime) + 1.15**2) / exptime

    return {
        "image_cube": image_cube,
        "error_cube": error_cube,
        "obs_time": ref_map.date.datetime,
        "wavelengths": wavelengths,
        "crop_info": {"center": crop_center, "size": crop_size},
    }


def create_synthetic_observation():
    """Create synthetic AIA-like observation for testing."""
    # Temperature grid
    temps = get_default_temperatures(n_bins=50)
    logt = np.log10(temps)

    # Create a known DEM: isothermal + broad component
    dem_true = (
        1e22 * np.exp(-0.5 * ((logt - 6.2) / 0.15) ** 2)  # Hot component
        + 5e21 * np.exp(-0.5 * ((logt - 5.9) / 0.3) ** 2)  # Warm component
    )

    # Get temperature response (will use fallback if aiapy not installed)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        response = get_temperature_response(temperatures=temps)

    # Compute synthetic intensities: I = integral(K * DEM * dT)
    dlogt = np.gradient(logt)
    dt = temps * np.log(10) * dlogt
    intensities = np.sum(response * dem_true[:, np.newaxis] * dt[:, np.newaxis], axis=0)

    # Add noise
    noise_level = 0.05
    errors = intensities * noise_level
    intensities_noisy = intensities + np.random.randn(6) * errors

    return {
        "temps": temps,
        "response": response,
        "dem_true": dem_true,
        "intensities": intensities_noisy,
        "errors": errors,
    }

## Synthetic Data Analysis

### 1. Creating Synthetic Observation

In [ ]:
data = create_synthetic_observation()
print(f"Temperature range: 10^{np.log10(data['temps'][0]):.1f} - "
      f"10^{np.log10(data['temps'][-1]):.1f} K")
print(f"Number of temperature bins: {len(data['temps'])}")
print(f"AIA wavelengths: {AIA_WAVELENGTHS} Å")
print(f"Intensities (DN/s): {data['intensities'].round(1)}")

### 2. Single-Pixel DEM Inversion

In [ ]:
dem, info = dem_sites_pixel(
    data["intensities"],
    data["errors"],
    data["response"],
    data["temps"],
    max_iter=100,
    tol=1e-4,
)

print(f"Converged: {info['converged']}")
print(f"Iterations: {info['iterations']}")
print(f"Chi-squared: {info['chi2']:.2f}")
print(f"DEM peak: {dem.max():.2e} cm^-5 K^-1")

# Derived quantities
em = get_emission_measure(dem, data["temps"])
t_mean = get_mean_temperature(dem, data["temps"])

print(f"\nDerived Quantities:")
print(f"Total Emission Measure: {em:.2e} cm^-5")
print(f"DEM-weighted Mean Temperature: {t_mean/1e6:.2f} MK")
print(f"Peak Temperature: {data['temps'][np.argmax(dem)]/1e6:.2f} MK")

### 3. Comparison with True DEM

In [ ]:
em_true = get_emission_measure(data["dem_true"], data["temps"])
t_mean_true = get_mean_temperature(data["dem_true"], data["temps"])
em_recovered = get_emission_measure(dem, data["temps"])
t_mean_recovered = get_mean_temperature(dem, data["temps"])

print(f"True EM: {em_true:.2e}, Recovered: {em_recovered:.2e}")
print(f"True T_mean: {t_mean_true/1e6:.2f} MK, Recovered: {t_mean_recovered/1e6:.2f} MK")

### 4. Small Map Processing Demo

In [ ]:
height, width = 16, 16
response = data["response"]

# Vary DEM spatially
y_grid, x_grid = np.meshgrid(np.arange(height), np.arange(width), indexing="ij")
temp_variation = 6.0 + 0.4 * np.sin(2 * np.pi * x_grid / width)

# Create image cube
image_cube = np.zeros((height, width, 6), dtype=np.float32)
temps = data["temps"]
logt = np.log10(temps)
dlogt = np.gradient(logt)
dt = temps * np.log(10) * dlogt

for i in range(height):
    for j in range(width):
        dem_local = 1e22 * np.exp(
            -0.5 * ((logt - temp_variation[i, j]) / 0.2) ** 2
        )
        image_cube[i, j] = np.sum(
            response * dem_local[:, np.newaxis] * dt[:, np.newaxis], axis=0
        )

error_cube = image_cube * 0.1

dem_cube, map_info = dem_map(
    image_cube, error_cube, response, temps,
    chunk_size=8, max_iter=50,
)

print(f"Image size: {height}x{width} pixels")
print(f"DEM cube shape: {dem_cube.shape}")
print(f"Mean iterations: {map_info['mean_iterations']:.1f}")

em_map_result = get_emission_measure(dem_cube, temps)
t_map = get_mean_temperature(dem_cube, temps)
print(f"T_mean range: {t_map.min()/1e6:.2f} - {t_map.max()/1e6:.2f} MK")

## Real AIA Data Analysis (Optional)

This section requires sunpy and network access.

In [ ]:
# Uncomment and run to download and analyze real AIA data
# This requires sunpy and network access

# if HAS_SUNPY:
#     obs_time = datetime(2024, 1, 15, 12, 0, 0)
#     print(f"Target time: {obs_time}")
#
#     files = download_aia_data(obs_time, data_dir="./data/aia_dem")
#
#     if len(files) == 6:
#         data = load_aia_cube(files, crop_size=128)
#         print(f"Image cube shape: {data['image_cube'].shape}")
#
#         temps = get_default_temperatures(n_bins=50)
#         response = get_temperature_response(
#             wavelengths=data["wavelengths"],
#             temperatures=temps,
#             time=data["obs_time"],
#         )
#
#         dem_cube, info = dem_map(
#             data["image_cube"],
#             data["error_cube"],
#             response,
#             temps,
#             chunk_size=64,
#             max_iter=50,
#         )
#
#         print(f"DEM cube shape: {dem_cube.shape}")

## Typical Usage Pattern

### Option 1: With real AIA data (recommended for research)

```python
from datetime import datetime
import astropy.units as u
from sunpy.net import Fido, attrs as a
from sunpy.map import Map
from egghouse.dem import     get_temperature_response, get_default_temperatures,     dem_map, get_emission_measure, get_mean_temperature, 

# Download AIA data
obs_time = datetime(2024, 1, 15, 12, 0, 0)
wavelengths = [94, 131, 171, 193, 211, 335]

for wave in wavelengths:
    result = Fido.search(
        a.Time(obs_time, obs_time),
        a.Instrument("AIA"),
        a.Wavelength(wave * u.angstrom),
    )
    files = Fido.fetch(result, path='./data/')

# Load and prepare data
maps = [Map(f) for f in sorted(files)]
image_cube = np.stack([m.data / m.exposure_time.value for m in maps], axis=-1)
error_cube = image_cube * 0.1  # Simple 10% error estimate

# Temperature grid and response
temps = get_default_temperatures(logt_min=5.5, logt_max=7.5, n_bins=100)
response = get_temperature_response(temperatures=temps, time=obs_time)

# Compute DEM
dem_cube, info = dem_map(image_cube, error_cube, response, temps, chunk_size=512)

# Derived quantities
em = get_emission_measure(dem_cube, temps)
t_mean = get_mean_temperature(dem_cube, temps)
```

### Option 2: With synthetic data (for testing)

```python
from egghouse.dem import dem_sites_pixel

intensities = np.array([10.0, 50.0, 200.0, 150.0, 80.0, 20.0])  # DN/s
errors = intensities * 0.1
dem, info = dem_sites_pixel(intensities, errors, response, temps)
```